# 04 — ML Data Loading

**Goal:** Load and organise the train/test BP and PPG datasets for the machine learning pipeline.

This notebook bridges the signal-processing pipeline (notebooks 01–03) and the ML pipeline (notebooks 05–06). It loads per-patient pickle files, aligns BP and PPG by patient ID, and saves combined structures ready for feature extraction.

**Inputs:** `data/raw/train/` and `data/raw/test/` (individual patient `.pkl` files)

**Outputs saved to `data/processed/`:**
- `ml_train.pkl` — {`bp`: dict, `ppg`: dict, `patient_ids`: list}
- `ml_test.pkl`  — same structure for the test split

---
### Pipeline position
```
01 Import → 02 Signal Processing → 03 Interpolation → [04 ML Data Loading] → 05 Features → 06 Models
```

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import timedelta
from scipy.interpolate import splrep, splev, CubicSpline

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
REPO_ROOT     = Path.cwd().parent if Path.cwd().name == 'improved' else Path.cwd().parent.parent
RAW_TRAIN_DIR = REPO_ROOT / 'data' / 'raw' / 'train'
RAW_TEST_DIR  = REPO_ROOT / 'data' / 'raw' / 'test'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Train data : {RAW_TRAIN_DIR}')
print(f'Test data  : {RAW_TEST_DIR}')

## 1. Loading Functions

All raw data follows the naming convention `<Type>_Table_<patient_id>.pkl`. A single reusable loader handles both BP and PPG files across train and test splits.

In [ ]:
def extract_patient_id(filename: str) -> str:
    """Return the numeric patient ID from 'BP_Table_12345678.pkl' → '12345678'."""
    return Path(filename).stem.split('_')[-1]


def load_split(data_dir: Path, prefix: str) -> dict:
    """
    Load all .pkl files in *data_dir* matching `<prefix>_Table_*.pkl`.

    Parameters
    ----------
    data_dir : directory containing the patient pickle files
    prefix   : 'BP' or 'PPG'

    Returns
    -------
    dict mapping patient_id (str) -> DataFrame
    """
    data = {}
    for path in sorted(data_dir.glob(f'{prefix}_Table_*.pkl')):
        pid = extract_patient_id(path.name)
        with open(path, 'rb') as f:
            df = pickle.load(f)
        if 'patient_id' not in df.columns:
            df.insert(0, 'patient_id', pid)
        data[pid] = df
    return data

## 2. Load Train & Test Data

In [ ]:
bp_train  = load_split(RAW_TRAIN_DIR, 'BP')
ppg_train = load_split(RAW_TRAIN_DIR, 'PPG')

bp_test   = load_split(RAW_TEST_DIR,  'BP')
ppg_test  = load_split(RAW_TEST_DIR,  'PPG')

print('── Train ──────────────────────────────')
print(f'  BP  recordings : {len(bp_train)}')
print(f'  PPG recordings : {len(ppg_train)}')
print()
print('── Test  ──────────────────────────────')
print(f'  BP  recordings : {len(bp_test)}')
print(f'  PPG recordings : {len(ppg_test)}')

## 3. Data Inspection

Quick look at the shape and columns for one patient to confirm the data structure is as expected.

In [ ]:
sample_pid = next(iter(bp_train))
print(f'Sample patient ID : {sample_pid}')
print()
print('BP DataFrame:')
print(bp_train[sample_pid].head())
print(f'Shape: {bp_train[sample_pid].shape}')
print()
if sample_pid in ppg_train:
    print('PPG DataFrame:')
    print(ppg_train[sample_pid].head())
    print(f'Shape: {ppg_train[sample_pid].shape}')

In [ ]:
# ── Overlap check: which patients have both BP and PPG? ────────────────────────
train_ids = sorted(set(bp_train) & set(ppg_train))
test_ids  = sorted(set(bp_test)  & set(ppg_test))

print(f'Train patients with both BP + PPG : {len(train_ids)}')
print(f'Test  patients with both BP + PPG : {len(test_ids)}')

## 4. Dataset Summary

Basic statistics on the blood pressure labels to understand the target distribution.

In [ ]:
# Collect all BP values from the training set
bp_columns = ['MAP', 'SBP', 'DBP']   # adjust to actual column names in your data
existing_bp_cols = [c for c in bp_columns
                    if any(c in df.columns for df in bp_train.values())]

if existing_bp_cols:
    all_bp = pd.concat(list(bp_train.values()), ignore_index=True)
    print(all_bp[existing_bp_cols].describe().round(1))

    fig, axes = plt.subplots(1, len(existing_bp_cols), figsize=(5 * len(existing_bp_cols), 4))
    if len(existing_bp_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, existing_bp_cols):
        ax.hist(all_bp[col].dropna(), bins=50, color='steelblue', edgecolor='none')
        ax.set_title(f'{col} distribution (train)')
        ax.set_xlabel('mmHg')
        ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('BP column names in your data:')
    print(next(iter(bp_train.values())).columns.tolist())
    print('Update bp_columns above to match your actual column names.')

## 5. Save Processed Data

In [ ]:
ml_train = {'bp': bp_train, 'ppg': ppg_train, 'patient_ids': train_ids}
ml_test  = {'bp': bp_test,  'ppg': ppg_test,  'patient_ids': test_ids}

for filename, obj in [('ml_train.pkl', ml_train), ('ml_test.pkl', ml_test)]:
    with open(PROCESSED_DIR / filename, 'wb') as f:
        pickle.dump(obj, f)
    print(f'Saved → data/processed/{filename}')